# 06 · Ideas that did not help

Two of them are re-run here from cached features; the rest are summarised with their results (their scripts are in
`archive/`). Every one was scored with the same photo-grouped 10 × 5-fold CV.

In [1]:
import numpy as np, pandas as pd
import common as c
pd.set_option('display.width', 200)
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

## A. Lean towards *Incomplete* (per-chore bias)

Most remaining dishes errors are dirty scenes called clean. A bias added to the *Incomplete* score of each chore shifts
the decision; its size is chosen inside each training fold (inner 5-fold CV), never on the photo being scored.

In [2]:
df = c.training_set(final=True); X = c.final_features(df.id); y = df.y.values
orig, ext = np.where(~df.external.values)[0], np.where(df.external.values)[0]
GRID = np.round(np.arange(-1, 1.0001, 0.05), 2)
def decide(S, bd, bt):
    S = S.copy(); S[:, 1] += bd; S[:, 3] += bt; return S.argmax(1)
def scores(tr, va):
    return c.proba(c.MODELS["RBF SVM"]().fit(X[tr], y[tr]), X[va])
P0, P1, B = np.zeros((c.REPEATS, len(y)), int), np.zeros((c.REPEATS, len(y)), int), []
for k, (tr, va) in enumerate(c.splits(df)):
    tro = tr[np.isin(tr, orig)]; S_in = np.zeros((len(tro), 4))
    for itr, iva in StratifiedKFold(5, shuffle=True, random_state=k).split(tro, y[tro]):
        S_in[iva] = scores(np.r_[tro[itr], ext], tro[iva])
    best = lambda col: max(((round(f1_score(y[tro], decide(S_in, *( (b, 0) if col == 1 else (0, b) )), average="macro"), 6), -abs(b), b) for b in GRID))[2]
    bd, bt = best(1), best(3); B.append((bd, bt))
    S = scores(tr, va); P0[k // 5, va] = S.argmax(1); P1[k // 5, va] = decide(S, bd, bt)
f = lambda P: np.mean([f1_score(y[orig], p[orig], average="macro") for p in P])
print(f"no bias {f(P0):.3f}   with bias {f(P1):.3f}   chosen bias: dishes {np.mean([b[0] for b in B]):+.2f}, trash {np.mean([b[1] for b in B]):+.2f}")

no bias 0.912   with bias 0.908   chosen bias: dishes -0.03, trash -0.03


## B. Remove the top 30% of every photo

Photo #2 has clean hanging pans at the top and the dirty bowl lower down; does the top mislead the model?
The same crop is applied to every photo (training and test).

In [3]:
import torch
from PIL import Image, ImageOps
z, pos = c._load("dinov2_l_bottom70")
missing = [int(i) for i in df.id if int(i) not in pos]
if missing:                                     # encode the bottom 70% of photos added since the cache was made
    from extract_features import load_model
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    enc, pre = load_model("dinov2_l", device)
    emb = dict(zip(z["ids"].tolist(), z["emb"]))
    with torch.no_grad():
        for i in missing:
            im = ImageOps.exif_transpose(Image.open(c.image_path(i))).convert("RGB"); w, h = im.size
            im = im.crop((0, int(h * 0.3), w, h))
            e = c.unit(enc(torch.stack([pre(im), pre(ImageOps.mirror(im))]).to(device)))
            emb[i] = c.unit(e[0] + e[1])
    keep = sorted(emb); np.savez(c.FEAT / "dinov2_l_bottom70.npz", ids=np.array(keep), emb=np.stack([emb[i] for i in keep]))
    z, pos = c._load("dinov2_l_bottom70")
bot = z["emb"][[pos[int(i)] for i in df.id]]
helpers = X[:, 2048:]
pd.concat([c.cross_validate(df, X, ("RBF SVM",), name="full photo (final)")[0],
           c.cross_validate(df, np.hstack([bot, helpers]), ("RBF SVM",), name="top 30% removed")[0]])

/Users/home/workspace/School/ML/competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,experiment,model,CV macro F1,std,dishes F1,trash F1
0,full photo (final),RBF SVM,0.912,0.013,0.893,0.931
0,top 30% removed,RBF SVM,0.877,0.020,0.854,0.902


## C. Summary of the other ideas

Numbers are from the training set and features in use when each idea was tried, so compare each row with its own baseline.

| idea | what it does | result (RBF SVM unless noted) |
|---|---|---|
| Fine-tune DINOv2 | train the last 4 transformer blocks on the photos (LP-FT, 448 px) | 0.856 = frozen features on the same folds; fixes #7, #99, breaks #81, #88 |
| Patch / tile classifiers | classify image patches or tiles, pool the votes (incl. NBNN) | ≤ 0.814 vs 0.812 baseline (66 photos) |
| Foreground only | keep the main object found by clustering patches | 0.767 avg, worse |
| Soiled-dish detector + rule | windows scored for food residue; "soiled anywhere ⇒ Incomplete" | fixes #2 half the time but breaks clean #7 and #50: 0.921 vs 0.926 |
| … with 40 Kaggle plates | add 20 clean + 20 dirty plate close-ups (MIT) to the detector | detector AUC 0.892 → 0.900, rule still 0.921 |
| Relabel by a sink rule | treat anything in the sink as Incomplete when training | 0.851 → 0.840–0.845 against the host labels |
| Ensembles / routing | average SVM, LR, MLP; or send dishes to a different model | within ±0.005 of the best single model |
| Crop of one photo only | replace only #2 by its bottom 70% | 0.935 → 0.927, #2 worse |

Scripts: `archive/ft_dinov2.py`, `archive/patch_models.py`, `archive/fg_features.py`, `archive/soiled_windows.py`,
`archive/priority_rule.py`, `archive/relabel_sink_cv.py`, `archive/ens_aug.py`, `archive/ens_open.py`.

**Result.** Neither bias nor cropping helps: the inner CV chooses a bias of about 0, and removing the top of the photos
lowers the score. The errors that remain (#26, #31, #60, #77, #101) are mostly photos whose host label disagrees with
how similar photos are labelled, which no feature change fixes.